In [13]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

# Puedes probar con RUC (11 dígitos) o con DNI (8 dígitos, ej: "44900952")
DOCUMENTO_PRUEBA = "20607524476"
URL_SUNAT = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

def limpiar(texto: str) -> str:
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()

def extraer_ficha_descriptiva(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    ficha = {}

    # 1. Capturar alerta tributaria si existe (ej: BAJA DE OFICIO)
    alerta = soup.select_one(".list-group-item-danger")
    if alerta:
        ficha["Alerta Tributaria"] = limpiar(alerta.get_text())

    # 2. Recorrer únicamente las filas descriptivas (ignorando contenedores con botones o inputs)
    for item in soup.select(".list-group-item"):
        if item.find(["button", "form", "input"]):
            continue

        for row in item.select(".row"):
            cols = row.find_all("div", recursive=False)

            # Filas de 4 columnas (ej: Fecha de Inscripción + Fecha de Inicio, Sistema Emisión + Comercio Exterior)
            if len(cols) == 4:
                t1, v1 = limpiar(cols[0].get_text()), limpiar(cols[1].get_text())
                t2, v2 = limpiar(cols[2].get_text()), limpiar(cols[3].get_text())
                if t1.endswith(":"):
                    ficha[t1.rstrip(":")] = v1
                if t2.endswith(":"):
                    ficha[t2.rstrip(":")] = v2

            # Filas de 2 columnas (Etiqueta + Valor simple o Tabla interna)
            elif len(cols) >= 2:
                t1 = limpiar(cols[0].get_text())
                if not t1.endswith(":"):
                    continue
                k = t1.rstrip(":")

                tabla_interna = cols[1].find("table")
                if tabla_interna:
                    lineas = [limpiar(tr.get_text()) for tr in tabla_interna.find_all("tr") if limpiar(tr.get_text())]
                    ficha[k] = lineas if len(lineas) > 1 else (lineas[0] if lineas else "-")
                else:
                    ficha[k] = limpiar(cols[1].get_text())

    return ficha

async def consultar_ficha_sunat(documento: str) -> dict:
    doc = documento.strip()
    if len(doc) not in (8, 11) or not doc.isdigit():
        return {"error": "El documento debe ser un DNI (8 dígitos) o RUC (11 dígitos) numérico."}

    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(
                headless=True,
                channel="chrome",
                args=["--disable-blink-features=AutomationControlled"]
            )
        except Exception:
            browser = await p.chromium.launch(
                headless=True,
                channel="msedge",
                args=["--disable-blink-features=AutomationControlled"]
            )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            viewport={"width": 1366, "height": 768},
            locale="es-PE"
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = await context.new_page()

        print(f"[1/3] Conectando a SUNAT para consultar {'RUC' if len(doc) == 11 else 'DNI'}: {doc}...")
        await page.goto(URL_SUNAT, wait_until="domcontentloaded", timeout=30000)

        if len(doc) == 11:
            await page.wait_for_selector("#txtRuc", timeout=10000)
            await page.fill("#txtRuc", doc)
            await page.click("#btnAceptar")
        else:
            await page.click("#btnPorDocumento")
            await page.wait_for_selector("#txtNumeroDocumento", timeout=10000)
            await page.fill("#txtNumeroDocumento", doc)
            await page.click("#btnAceptar")

            enlace_ruc = "a.aRuc, .list-group-item a[data-ruc], a:has-text('10'), a:has-text('15'), a:has-text('17')"
            try:
                await page.wait_for_selector("text=Número de RUC:, " + enlace_ruc, timeout=15000)
                if await page.query_selector(enlace_ruc) and not await page.query_selector("text=Número de RUC:"):
                    await page.click(enlace_ruc)
            except Exception:
                pass

        print("[2/3] Esperando renderizado de la ficha descriptiva...")
        await page.wait_for_selector("text=Número de RUC:", timeout=15000)

        html_ficha = await page.content()
        ficha_extraida = extraer_ficha_descriptiva(html_ficha)
        print(f"[3/3] Extracción exitosa ({len(ficha_extraida)} campos obtenidos).")

        await browser.close()
        return ficha_extraida

# Ejecución en el notebook
ficha = await consultar_ficha_sunat(DOCUMENTO_PRUEBA)
print("\n================== FICHA DESCRIPTIVA SUNAT ==================")
for k, v in ficha.items():
    print(f"- {k}: {json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v}")

[1/3] Conectando a SUNAT para consultar RUC: 20607524476...
[2/3] Esperando renderizado de la ficha descriptiva...
[3/3] Extracción exitosa (19 campos obtenidos).

================== FICHA DESCRIPTIVA SUNAT ==================
- Alerta Tributaria: IMPORTANTE: Los comprobantes de pago o notas de débito emitidos por este contribuyente no dan derecho a crédito fiscal del IGV, en tanto se encuentra con estado de BAJA DE OFICIO
- Número de RUC: 20607524476 - INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA
- Tipo Contribuyente: EMPRESA INDIVIDUAL DE RESP. LTDA
- Nombre Comercial: INVERSIONES HUAYNA DEL PERU
- Fecha de Inscripción: 24/02/2021
- Fecha de Inicio de Actividades: 15/03/2021
- Estado del Contribuyente: BAJA DE OFICIO Fecha de Baja: 30/04/2026
- Condición del Contribuyente: HABIDO
- Domicilio Fiscal: CAL.VILLA LA RINCONADA LOTE. H-1 APV. VILLA LA RINCONADA CUSCO - CUSCO - SAN JERONIMO
- Sistema Emisión de Comprobante: COMPUTARIZADO
- Actividad 

In [1]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

RUC_PRUEBA = "20607524476"
URL_INICIO = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

def limpiar(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip() if t else ""

def parse_representantes(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    reps = []
    for tr in soup.find_all("tr"):
        cols = [limpiar(td.get_text()) for td in tr.find_all("td")]
        # Validar que tenga las 5 columnas de la tabla (Documento, Nro, Nombre, Cargo, Fecha)
        if len(cols) >= 5 and not cols[0].lower().startswith("documento"):
            reps.append({
                "document_type": cols[0],
                "document_number": cols[1],
                "full_name": cols[2],
                "role": cols[3],
                "start_date": cols[4]
            })
    return reps

def parse_establecimientos(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    anexos = []
    for tr in soup.find_all("tr"):
        cols = [limpiar(td.get_text()) for td in tr.find_all("td")]
        # Validar las 4 columnas (Código, Tipo, Dirección, Actividad)
        if len(cols) >= 4 and not cols[0].lower().startswith("código") and not cols[0].lower().startswith("codigo"):
            anexos.append({
                "code": cols[0],
                "establishment_type": cols[1],
                "address": cols[2],
                "economic_activity": cols[3]
            })
    return anexos

async def extraer_dos_botones_sunat(ruc: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            channel="chrome",
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            viewport={"width": 1366, "height": 768},
            locale="es-PE"
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = await context.new_page()

        print(f"[1/3] Ingresando RUC {ruc} en FrameCriterioBusquedaWeb.jsp...")
        await page.goto(URL_INICIO, wait_until="domcontentloaded", timeout=25000)
        await page.fill("#txtRuc", ruc)
        await page.click("#btnAceptar")

        # Esperar a que llegue a jcrS00Alias y carguen los botones inferiores
        await page.wait_for_selector("button:has-text('Representante')", timeout=15000)
        print(f"[2/3] Enlace actual: {page.url}")

        # Función JS que intercepta el POST hacia jcrS00Alias de cualquier botón sin recargar la pestaña
        js_extraer_boton = """
        async (textoBoton) => {
            const botones = Array.from(document.querySelectorAll('button, input[type="button"], input[type="submit"]'));
            const btn = botones.find(b => (b.innerText || b.value || '').includes(textoBoton));
            if (!btn) return "";

            return new Promise((resolve) => {
                const origSubmit = HTMLFormElement.prototype.submit;
                
                // Interceptar el envío del formulario cuando el botón sea presionado
                HTMLFormElement.prototype.submit = function() {
                    const formData = new URLSearchParams(new FormData(this)).toString();
                    const actionUrl = this.action || 'jcrS00Alias';
                    HTMLFormElement.prototype.submit = origSubmit;
                    
                    fetch(actionUrl, {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/x-www-form-urlencoded' },
                        body: formData
                    })
                    .then(r => r.arrayBuffer())
                    .then(buf => {
                        const html = new TextDecoder('iso-8859-1').decode(buf);
                        resolve(html);
                    })
                    .catch(() => resolve(""));
                };

                btn.click();

                setTimeout(() => {
                    HTMLFormElement.prototype.submit = origSubmit;
                    resolve("");
                }, 5000);
            });
        }
        """

        print("[3/3] Consultando botones de Representantes Legales y Establecimientos Anexos...")
        html_reps = await page.evaluate(js_extraer_boton, "Representante")
        html_anexos = await page.evaluate(js_extraer_boton, "Establecimiento")

        representantes = parse_representantes(html_reps)
        establecimientos = parse_establecimientos(html_anexos)

        await browser.close()

        return {
            "legal_representatives": representantes,
            "branch_offices": establecimientos
        }

# Ejecutar en la celda
resultado = await extraer_dos_botones_sunat(RUC_PRUEBA)
print("\n================ RESULTADO DE LOS 2 BOTONES ================")
print(json.dumps(resultado, indent=2, ensure_ascii=False))

[1/3] Ingresando RUC 20607524476 en FrameCriterioBusquedaWeb.jsp...
[2/3] Enlace actual: https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/jcrS00Alias
[3/3] Consultando botones de Representantes Legales y Establecimientos Anexos...

================ RESULTADO DE LOS 2 BOTONES ================
{
  "legal_representatives": [
    {
      "document_type": "DNI",
      "document_number": "44900952",
      "full_name": "RUIZ AUCCAPIÑA RONALD",
      "role": "TITULAR-GERENTE",
      "start_date": "08/02/2021"
    }
  ],
  "branch_offices": [
    {
      "code": "0004",
      "establishment_type": "LO. L. COMERCIAL",
      "address": "LOTE. 105 GRU. MERCADO MAYORISTA FERIAL DE PRODUCTORES OTR. STAND 105 CUSCO - CUSCO - SAN JERONIMO",
      "economic_activity": "4630"
    },
    {
      "code": "0005",
      "establishment_type": "DE. DEPOSITO",
      "address": "CAR.SANTA MARIA NRO. SN APV. APV. SANTA MARIA CUSCO - CUSCO - SAYLLA",
      "economic_activity": "4630"
    }
  ]
}
